In [1]:
!pip install -q transformers datasets scikit-learn

In [2]:
import os
import re
import pandas as pd
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# =========================
# 공통 설정
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 256

# RI: 1 = 탐구형(I)
AXIS_THRESHOLDS = {
    "RI": 0.56,
}

print("device:", device)
print("MODEL_NAME:", MODEL_NAME)
print("MAX_LENGTH:", MAX_LENGTH)
print("AXIS_THRESHOLDS:", AXIS_THRESHOLDS)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

device: cuda
MODEL_NAME: klue/bert-base
MAX_LENGTH: 256
AXIS_THRESHOLDS: {'RI': 0.56}


In [3]:
def train_axis_model(csv_path, save_dir):
    df = pd.read_csv(csv_path).dropna()
    df.columns = ['review_text', 'label']
    df['label'] = df['label'].astype(int)

    # 저장 폴더 미리 생성
    os.makedirs(save_dir, exist_ok=True)

    # ✅ 1. train/validation split을 여기서 한 번만 만든다
    train_df, val_df = train_test_split(
        df,
        test_size=0.2,
        random_state=42,
        stratify=df['label']   # A/E 비율이 train과 val에 비슷하게 들어가도록 함
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    # ✅ 2. 나중에 오분류 분석할 때 같은 검증셋을 쓰기 위해 저장
    train_df.to_csv(f"{save_dir}/train_split.csv", index=False, encoding="utf-8-sig")
    val_df.to_csv(f"{save_dir}/val_split.csv", index=False, encoding="utf-8-sig")

    print(f"학습셋 크기: {len(train_df)}개")
    print(f"검증셋 크기: {len(val_df)}개")
    print("검증셋 라벨 분포")
    print(val_df['label'].value_counts().sort_index())

    # ✅ 3. class weight는 전체 df가 아니라 train_df 기준으로 계산하는 게 더 안전함
    class_weights = compute_class_weight(
        'balanced',
        classes=np.unique(train_df['label']),
        y=train_df['label']
    )
    weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

    # ✅ 4. sklearn으로 나눈 train_df, val_df를 HuggingFace Dataset으로 변환
    train_dataset = Dataset.from_pandas(train_df)
    val_dataset = Dataset.from_pandas(val_df)

    def tokenize_fn(x):
        return tokenizer(
            x["review_text"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH
        )

    tokenized_train = train_dataset.map(tokenize_fn, batched=True)
    tokenized_val = val_dataset.map(tokenize_fn, batched=True)

    def compute_metrics(pred):
        labels = pred.label_ids
        preds = pred.predictions.argmax(-1)

        return {
            "accuracy": accuracy_score(labels, preds),
            "f1_macro": f1_score(labels, preds, average="macro"),
            "f1_binary": f1_score(labels, preds, average="binary"),
            "balanced_accuracy": balanced_accuracy_score(labels, preds)
        }

    class CustomTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            loss_fct = nn.CrossEntropyLoss(weight=weights_tensor)
            loss = loss_fct(outputs.logits.view(-1, 2), labels.view(-1))
            return (loss, outputs) if return_outputs else loss

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    ).to(device)

    args = TrainingArguments(
      output_dir=save_dir,
      eval_strategy="epoch",
      save_strategy="epoch",
      learning_rate=2e-5,
      num_train_epochs=4,
      per_device_train_batch_size=16,
      load_best_model_at_end=True,
      metric_for_best_model="f1_macro",
      greater_is_better=True
  )

    trainer = CustomTrainer(
        model=model,
        args=args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        compute_metrics=compute_metrics
    )

    trainer.train()

    trainer.save_model(save_dir)
    tokenizer.save_pretrained(save_dir)

    print(f"✅ 모델 저장 완료: {save_dir}")
    print(f"✅ 검증셋 저장 완료: {save_dir}/val_split.csv")


In [4]:
# =========================
# RI 모델만 학습
# =========================
RI_CSV_PATH = "/content/train_RI.csv"
RI_result_df_SAVE_DIR = "./model_axis2_RI"

train_axis_model(
    RI_CSV_PATH,
    RI_result_df_SAVE_DIR
)

학습셋 크기: 2400개
검증셋 크기: 600개
검증셋 라벨 분포
label
0    300
1    300
Name: count, dtype: int64


Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on you

... (4 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ 모델 저장 완료: ./model_axis2_RI
✅ 검증셋 저장 완료: ./model_axis2_RI/val_split.csv


In [5]:
# =========================
# RI 모델 로드
# =========================
model_ri = AutoModelForSequenceClassification.from_pretrained("./model_axis2_RI").to(device)
model_ri.eval()

RI_THRESHOLD = AXIS_THRESHOLDS["RI"]

print("✅ RI 모델 로드 완료")
print("RI_THRESHOLD:", RI_THRESHOLD)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ RI 모델 로드 완료
RI_THRESHOLD: 0.56


In [6]:
# =========================
# RI 문장 단위 단서 기반 보정 규칙
# 기존 방식 유지: 모델 예측 + 문장 단위 cue 분석 + R/I 보정
# =========================

R_CUES = [
    "이해할 수 있었다", "이해하기 쉬웠다", "이해 됐다", "잘 설명", "잘 정리",
    "도움이 되었다", "유익했다", "납득", "공감할 수 있었다",
    "받아들일 수 있었다", "설득력 있었다", "좋은 내용",
    "명확하다", "쉽게 설명", "알 수 있었다", "배울 수 있었다",
    "필요한 책", "추천하고 싶다", "의미 있었다", "잘 전달",
    "핵심을 알려준다", "중요성을 알게 되었다", "설명한다"
]

I_CUES = [
    "왜", "의문", "궁금", "질문", "생각해볼", "고민하게",
    "다른 관점", "다르게 볼", "정말", "과연", "항상 맞는지",
    "적용될 수 있는지", "한계", "비판", "반문", "의심",
    "더 알고 싶다", "더 생각", "다른 해석", "가능할까",
    "타당한지", "맞는 말인지", "확실하지 않다"
]


def split_sentences(text):
    sentences = re.split(r'(?<=[.!?。！？])\s+', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences if sentences else [text]


def count_cues(text, cues):
    return sum(1 for cue in cues if cue in text)


def get_ri_base_prediction(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    ).to(device)

    with torch.no_grad():
        outputs = model_ri(**inputs)

    probs = F.softmax(outputs.logits, dim=-1)[0]
    prob_R = probs[0].item()
    prob_I = probs[1].item()
    base_pred = 1 if prob_I >= RI_THRESHOLD else 0

    return base_pred, prob_R, prob_I


def analyze_ri_sentences(text):
    sentences = split_sentences(text)
    rows = []

    for idx, sent in enumerate(sentences, start=1):
        r_count = count_cues(sent, R_CUES)
        i_count = count_cues(sent, I_CUES)

        if r_count > i_count:
            cue_label = "수용형(R)"
        elif i_count > r_count:
            cue_label = "탐구형(I)"
        else:
            cue_label = "중립/불명확"

        rows.append({
            "문장번호": idx,
            "문장": sent,
            "R_cue_count": r_count,
            "I_cue_count": i_count,
            "문장단서판단": cue_label,
        })

    return pd.DataFrame(rows)


def predict_ri_with_context_rule(text):
    """
    R/I 전용 보정 예측.
    기본 모델은 그대로 쓰되,
    문장 단위 cue 분석을 통해 수용/탐구 중심성을 보정한다.
    """

    # 1. 전체 리뷰 기준 모델 예측
    base_pred, prob_R, prob_I = get_ri_base_prediction(text)

    # 2. 문장 단위 cue 분석
    sentence_df = analyze_ri_sentences(text)

    r_sentence_count = (sentence_df["문장단서판단"] == "수용형(R)").sum()
    i_sentence_count = (sentence_df["문장단서판단"] == "탐구형(I)").sum()

    total_r_cues = count_cues(text, R_CUES)
    total_i_cues = count_cues(text, I_CUES)

    # 3. 보정 규칙

    # 모델은 I라고 했지만 → R로 보정
    if base_pred == 1:
        if (
            r_sentence_count >= 2
            and total_r_cues >= total_i_cues + 2
            and total_r_cues >= 4
        ):
            corrected_pred = 0
            correction_reason = "R/I 보정: 책의 주장·내용을 받아들이는 수용 서술이 리뷰 중심"
        else:
            corrected_pred = base_pred
            correction_reason = "모델 예측 유지"

    # 모델은 R이라고 했지만 → I로 보정
    elif base_pred == 0:
        if (
            i_sentence_count >= 2
            and total_i_cues >= total_r_cues + 2
            and total_i_cues >= 4
        ):
            corrected_pred = 1
            correction_reason = "R/I 보정: 질문·의문·확장적 사고가 리뷰 중심"
        else:
            corrected_pred = base_pred
            correction_reason = "모델 예측 유지"

    else:
        corrected_pred = base_pred
        correction_reason = "모델 예측 유지"

    conf = prob_I if corrected_pred == 1 else prob_R

    return {
        "pred": corrected_pred,
        "label": "수용형(R)" if corrected_pred == 0 else "탐구형(I)",
        "confidence": conf,
        "prob_R": prob_R,
        "prob_I": prob_I,
        "base_pred": base_pred,
        "base_label": "수용형(R)" if base_pred == 0 else "탐구형(I)",
        "correction_reason": correction_reason,
        "r_sentence_count": int(r_sentence_count),
        "i_sentence_count": int(i_sentence_count),
        "total_r_cues": int(total_r_cues),
        "total_i_cues": int(total_i_cues),
        "sentence_df": sentence_df,
    }

In [7]:
# =========================
# 보기 좋은 단일 리뷰 출력 함수
# =========================

def display_ri_prediction(text):
    result = predict_ri_with_context_rule(text)

    summary_df = pd.DataFrame([{
        "최종 예측": result["label"],
        "최종 확신도": f"{result['confidence'] * 100:.1f}%",
        "수용형(R) 확률": f"{result['prob_R'] * 100:.1f}%",
        "탐구형(I) 확률": f"{result['prob_I'] * 100:.1f}%",
        "모델 기본 예측": result["base_label"],
        "보정 사유": result["correction_reason"],
        "수용형 문장 수": result["r_sentence_count"],
        "탐구형 문장 수": result["i_sentence_count"],
        "R 단서 수": result["total_r_cues"],
        "I 단서 수": result["total_i_cues"],
    }])

    print("=" * 80)
    print("입력 리뷰")
    print("=" * 80)
    print(text.strip())

    print("\n" + "=" * 80)
    print("SI 최종 예측 요약")
    print("=" * 80)
    display(summary_df)

    print("\n" + "=" * 80)
    print("문장 단위 분석")
    print("=" * 80)
    display(result["sentence_df"])

    return result


In [15]:
# =========================
# 테스트 리뷰
# =========================

test_review = """
저자가 제시하는 주장에 대해 계속 생각하게 되었고, 왜 이런 결론이 나왔는지에 대한 궁금증이 생겼다. 특히 다른 관점에서는 어떻게 해석될 수 있을지도 고민하게 되었다. 다만 전반적인 설명 자체는 이해하기 어렵지 않았다.
"""

result = display_ri_prediction(test_review)


입력 리뷰
저자가 제시하는 주장에 대해 계속 생각하게 되었고, 왜 이런 결론이 나왔는지에 대한 궁금증이 생겼다. 특히 다른 관점에서는 어떻게 해석될 수 있을지도 고민하게 되었다. 다만 전반적인 설명 자체는 이해하기 어렵지 않았다.

SI 최종 예측 요약


,최종 예측,최종 확신도,수용형(R) 확률,탐구형(I) 확률,모델 기본 예측,보정 사유,수용형 문장 수,탐구형 문장 수,R 단서 수,I 단서 수
0,탐구형(I),12.5%,87.5%,12.5%,수용형(R),R/I 보정: 질문·의문·확장적 사고가 리뷰 중심,0,2,0,4



문장 단위 분석


,문장번호,문장,R_cue_count,I_cue_count,문장단서판단
0,1,"저자가 제시하는 주장에 대해 계속 생각하게 되었고, 왜 이런 결론이 나왔는지에 대한...",0,2,탐구형(I)
1,2,특히 다른 관점에서는 어떻게 해석될 수 있을지도 고민하게 되었다.,0,2,탐구형(I)
2,3,다만 전반적인 설명 자체는 이해하기 어렵지 않았다.,0,0,중립/불명확


In [9]:
# =========================
# RI 검증셋 성능 / 오분류 요약
# =========================

def predict_ri_batch(texts, batch_size=32, max_length=MAX_LENGTH):
    model_ri.eval()

    preds = []
    probs_R = []
    probs_I = []
    confidences = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length
        ).to(device)

        with torch.no_grad():
            logits = model_ri(**inputs).logits

        prob = F.softmax(logits, dim=-1).cpu()
        prob_R = prob[:, 0]
        prob_I = prob[:, 1]

        pred = (prob_R >= RI_THRESHOLD).long()
        confidence = torch.where(pred == 1, prob_R, prob_I)

        preds.extend(pred.tolist())
        probs_R.extend(prob_R.tolist())
        probs_I.extend(prob_I.tolist())
        confidences.extend(confidence.tolist())

    return preds, probs_R, probs_I, confidences


def analyze_ri_validation(val_csv_path="./model_axis2_RI/val_split.csv", save_dir="/content"):
    print("\n" + "=" * 70)
    print("RI 축 성능 / 오분류 요약")
    print("=" * 70)

    val_df = pd.read_csv(val_csv_path)
    val_df["label"] = val_df["label"].astype(int)

    label_counts = val_df["label"].value_counts().sort_index()
    print("\n[1] 검증셋 라벨 분포")
    print(f"  수용형(R=0): {label_counts.get(0, 0)}개")
    print(f"  탐구형(I=1): {label_counts.get(1, 0)}개")
    print(f"  전체: {len(val_df)}개")

    texts = val_df["review_text"].tolist()
    preds, probs_R, probs_I, confidences = predict_ri_batch(texts)

    val_df["pred"] = preds
    val_df["prob_R"] = probs_R
    val_df["prob_I"] = probs_I
    val_df["confidence"] = confidences
    val_df["correct"] = val_df["label"] == val_df["pred"]

    y_true = val_df["label"]
    y_pred = val_df["pred"]

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    print("\n[2] 전체 성능")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Macro F1 : {macro_f1:.4f}")

    print("\n[3] Confusion Matrix")
    print("       예측 R    예측 I")
    print(f"실제 R   {cm[0,0]:>5}    {cm[0,1]:>5}")
    print(f"실제 I   {cm[1,0]:>5}    {cm[1,1]:>5}")

    print("\n[4] 오분류 방향")
    print(f"  수용형(R) → 탐구형(I): {cm[0,1]}개")
    print(f"  탐구형(I) → 수용형(R): {cm[1,0]}개")

    print("\n[5] 상세 리포트")
    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["수용형(R)", "탐구형(I)"],
        digits=4
    ))

    os.makedirs(save_dir, exist_ok=True)
    result_path = f"{save_dir}/prediction_result_RI.csv"
    val_df.to_csv(result_path, index=False, encoding="utf-8-sig")

    print("\n[6] 저장 완료")
    print(f"  전체 예측 결과: {result_path}")

    return val_df


ri_result_df = analyze_ri_validation()



RI 축 성능 / 오분류 요약

[1] 검증셋 라벨 분포
  수용형(R=0): 300개
  탐구형(I=1): 300개
  전체: 600개

[2] 전체 성능
  Accuracy : 0.1733
  Macro F1 : 0.1713

[3] Confusion Matrix
       예측 R    예측 I
실제 R      37      263
실제 I     233       67

[4] 오분류 방향
  수용형(R) → 탐구형(I): 263개
  탐구형(I) → 수용형(R): 233개

[5] 상세 리포트
              precision    recall  f1-score   support

      수용형(R)     0.1370    0.1233    0.1298       300
      탐구형(I)     0.2030    0.2233    0.2127       300

    accuracy                         0.1733       600
   macro avg     0.1700    0.1733    0.1713       600
weighted avg     0.1700    0.1733    0.1713       600


[6] 저장 완료
  전체 예측 결과: /content/prediction_result_RI.csv


In [10]:
# =========================
# 여러 리뷰를 한 번에 예측하고 표로 보기
# =========================

def predict_ri_reviews_to_table(reviews):
    rows = []
    for i, text in enumerate(reviews, start=1):
        result = predict_ri_with_context_rule(text)
        rows.append({
            "번호": i,
            "리뷰": text.strip().replace("\n", " ")[:120],
            "최종 예측": result["label"],
            "수용형(R) 확률": result["prob_R"],
            "탐구형(I) 확률": result["prob_I"],
            "보정 사유": result["correction_reason"],
            "분석형 문장 수": result["a_sentence_count"],
            "공감형 문장 수": result["e_sentence_count"],
        })
    return pd.DataFrame(rows)

sample_reviews = [
    "이 책은 인간관계의 원리를 사례를 통해 체계적으로 설명한다. 각 장의 구성도 명확하고 핵심 개념도 잘 정리되어 있다.",
    "읽는 내내 내가 겪었던 일이 떠올라서 마음이 아팠다. 주인공의 감정이 너무 잘 느껴져서 깊이 공감했다.",
    "이 책은 인간관계의 원리를 사례를 통해 체계적으로 설명한다. 각 장의 구성이 명확하다. 다만 내 경험이 떠올라 조금 씁쓸했다.",
]

sample_result_df = predict_ri_reviews_to_table(sample_reviews)
display(sample_result_df)


KeyError: 'prob_S'